In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [3]:
ppi = pd.read_csv("../../data/Final/kidney_PPI_final.csv")
labels = pd.read_csv("../../data/Final/labels.csv")
expressions = pd.read_csv("../../data/Final/expressions.csv")
mutations = pd.read_csv("../../data/Final/mutations.csv")

In [4]:
ppi_genes = pd.concat([ppi['A'], ppi['B']]).unique()

In [5]:
expr_no_id = expressions.drop('ModelID', axis=1)
mut_no_id = mutations.drop('ModelID', axis=1)
mutations_aligned = mut_no_id.reindex(columns=ppi_genes, fill_value=0)
labels_no_id = labels.drop('ModelID', axis=1)
labels_reindexed = labels_no_id.reindex(columns=ppi_genes)


In [6]:
labels_reindexed.isna().sum().sum()


np.int64(32147)

In [7]:
x_list = []
y_list = []
mask_list = []
for i in range(32):
    expr_row = expr_no_id[ppi_genes].iloc[i].values
    mut_row = mutations_aligned.iloc[i].values
    x = torch.tensor(np.column_stack([expr_row, mut_row]), dtype=torch.float)
    y_raw = labels_reindexed.iloc[i]
    mask_i = torch.tensor((~y_raw.isna().values) & np.isin(ppi_genes, labels_no_id.columns), dtype=torch.bool)
    y = torch.tensor(y_raw.fillna(0).values, dtype=torch.float)

    x_list.append(x)
    y_list.append(y)
    mask_list.append(mask_i)

In [8]:
import random
random.seed(12)
combined = list(zip(x_list, y_list, mask_list))
random.shuffle(combined)
x_list, y_list, mask_list = zip(*combined)
train_x, train_y = list(x_list[:26]), list(y_list[:26])
test_x, test_y = list(x_list[26:]), list(y_list[26:])
train_m, test_m = list(mask_list[:26]),list(mask_list[26:])

In [9]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 64)
        self.relu = nn.LeakyReLU()
        self.dropout = nn.Dropout(0.3)
        self.layer2 = nn.Linear(64, 1)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer2(x)
        return x.squeeze()


In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
loss_fn = nn.MSELoss()


In [11]:
epochs = 200
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x, y, m in zip(train_x, train_y, train_m):
        x, y, m = x.to(device), y.to(device), m.to(device)
        pred = model(x)
        loss = loss_fn(pred[m], y[m])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_x):.4f}")


Epoch 10, Loss: 0.1790
Epoch 20, Loss: 0.1661
Epoch 30, Loss: 0.1652
Epoch 40, Loss: 0.1648
Epoch 50, Loss: 0.1645
Epoch 60, Loss: 0.1642
Epoch 70, Loss: 0.1641
Epoch 80, Loss: 0.1640
Epoch 90, Loss: 0.1640
Epoch 100, Loss: 0.1637
Epoch 110, Loss: 0.1637
Epoch 120, Loss: 0.1637
Epoch 130, Loss: 0.1636
Epoch 140, Loss: 0.1636
Epoch 150, Loss: 0.1635
Epoch 160, Loss: 0.1635
Epoch 170, Loss: 0.1634
Epoch 180, Loss: 0.1633
Epoch 190, Loss: 0.1635
Epoch 200, Loss: 0.1635


In [12]:
model.eval()
test_losses = []
with torch.no_grad():
    for x, y, m in zip(test_x, test_y, test_m):
        x, y, m = x.to(device), y.to(device), m.to(device)
        pred = model(x)
        loss = loss_fn(pred[m], y[m])
        test_losses.append(loss.item())
print(f"Test MSE: {np.mean(test_losses):.4f}")


Test MSE: 0.1610


In [13]:
from scipy.stats import pearsonr
correlations = []
with torch.no_grad():
    for x, y, m in zip(test_x, test_y, test_m):
        x, y, m = x.to(device), y.to(device), m.to(device)
        pred = model(x)
        p = pred[m].cpu().numpy()
        a = y[m].cpu().numpy()
        r, _ = pearsonr(p, a)
        correlations.append(r)
print(f"Test Pearson r: {np.mean(correlations):.4f}")

Test Pearson r: 0.3933
